# Step 3 — Build FAISS Index (Candidate Generation, Embedding Side)

**What changed vs. the version you gave me, and why:**

1. **Hard GPU dependency with no fallback.** `faiss.StandardGpuResources()` doesn't even *exist* on the
   `faiss-cpu` build (the one pinned in requirements.txt) — calling it crashes with an `AttributeError`,
   not a graceful failure. Detects GPU availability properly and falls back to CPU automatically. If you
   do have a GPU and want GPU-accelerated build/search, you need `faiss-gpu` installed instead of
   `faiss-cpu` (can't have both) — this notebook adapts to whichever you actually have.
2. **Always brute-force `IndexFlatIP`.** Fine for a small test-time index, but exact search against
   several million vectors per query is the classic scalability wall at 50L+ scale. Now auto-switches to
   an IVF (approximate) index above a configurable size threshold, with Flat kept for smaller indexes
   where exact search is still cheap and safest.
3. **File-glob mismatch with the updated Step 2 output.** The Step 2 notebook now saves
   `part_*_meta.parquet` (plus separate `.npz` TF-IDF files), not plain `part_*.parquet` — the loader
   here is updated to match, and prefers the consolidated `embeddings.npy` from Step 2's optional
   consolidation cell when it's available (much lighter than reading list-columns out of parquet at
   scale).
4. **No integrity checks before `vstack`.** A NaN or a malformed row from Step 2 would previously fail
   deep inside FAISS with a cryptic error. Now checked explicitly, with a clear message pointing back at
   Step 2.
5. **Index metadata saved alongside the index** (`*_info.json` — kind, nlist, nprobe, dim, vector count)
   so a downstream search step knows how to load and query it correctly, rather than needing to guess.
6. **Added Step 3b (not in your original): actually querying the index.** Building an index doesn't
   produce candidates by itself — this is flagged clearly as an addition. Delete that section if you
   already have a separate script for it.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import faiss

In [ ]:
# =====================================================
# CONFIG
# =====================================================

TRAIN_DIRS = [
    Path("processed/source2_train_embeddings"),
    Path("processed/source3_train_embeddings"),
]

TEST_DIRS = [
    Path("processed/source2_test_embeddings"),
    Path("processed/source3_test_embeddings"),
]

OUT_DIR = Path("faiss")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Above this many vectors, switch from exact (Flat) to approximate (IVF)
# search - brute-force cosine search against several million vectors per
# query is the scalability bottleneck at 50L+ scale, not index building.
IVF_THRESHOLD = 200_000
IVF_NLIST_MAX = 4096
IVF_NPROBE = 32
IVF_TRAIN_SAMPLE_MIN = 200_000

In [ ]:
# =====================================================
# GPU AVAILABILITY
# =====================================================

def detect_gpu():
    has_gpu_api = hasattr(faiss, "StandardGpuResources") and hasattr(faiss, "get_num_gpus")
    if not has_gpu_api:
        return False
    try:
        return faiss.get_num_gpus() > 0
    except Exception:
        return False

USE_GPU = detect_gpu()
print(f"FAISS GPU support: {'available' if USE_GPU else 'not available - building on CPU'}")

In [ ]:
# =====================================================
# LOAD EMBEDDINGS
# =====================================================
# Prefers the consolidated embeddings.npy + entity_id_order.txt from the
# Step 2 notebook's optional consolidation cell. Falls back to the
# per-chunk *_meta.parquet files if that hasn't been run.

def load_embeddings_and_ids(input_dirs):
    all_ids = []
    all_embeddings = []

    for folder in input_dirs:
        npy_path = folder / "embeddings.npy"
        id_path = folder / "entity_id_order.txt"

        if npy_path.exists() and id_path.exists():
            print(f"  {folder}: using consolidated array")
            all_embeddings.append(np.load(npy_path).astype(np.float32))
            all_ids.extend(id_path.read_text(encoding="utf-8").splitlines())
            continue

        print(f"  {folder}: no consolidated array - reading per-chunk parquet "
              f"(consider running the Step 2 consolidation cell for large sources)")
        meta_files = sorted(folder.glob("part_*_meta.parquet"))
        if not meta_files:
            raise FileNotFoundError(
                f"No 'part_*_meta.parquet' files found in {folder}. If these came from "
                f"an older version of the Step 2 script, the filenames won't match - "
                f"re-run the updated Step 2 notebook first."
            )
        for file in meta_files:
            df = pd.read_parquet(file, columns=["entity_id", "embedding"])
            all_ids.extend(df["entity_id"].tolist())
            all_embeddings.append(np.vstack(df["embedding"].tolist()).astype(np.float32))

    embeddings = np.vstack(all_embeddings).astype(np.float32)
    return embeddings, all_ids


def validate_embeddings(embeddings, name):
    if np.isnan(embeddings).any():
        raise ValueError(f"[{name}] NaNs found in embeddings - check the Step 2 output before indexing.")
    if np.isinf(embeddings).any():
        raise ValueError(f"[{name}] Infs found in embeddings - check the Step 2 output before indexing.")
    print(f"[{name}] {embeddings.shape[0]:,} vectors x {embeddings.shape[1]} dims - integrity check passed.")

In [ ]:
# =====================================================
# BUILD INDEX (auto Flat vs IVF, GPU with safe CPU fallback)
# =====================================================

def build_faiss_index(embeddings, use_gpu):
    n, dim = embeddings.shape
    use_ivf = n > IVF_THRESHOLD

    if use_ivf:
        nlist = int(min(IVF_NLIST_MAX, max(64, 4 * np.sqrt(n))))
        quantizer = faiss.IndexFlatIP(dim)
        cpu_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    else:
        nlist = None
        cpu_index = faiss.IndexFlatIP(dim)

    working_index = cpu_index
    gpu_res = None
    if use_gpu:
        try:
            gpu_res = faiss.StandardGpuResources()
            working_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
        except Exception as e:
            print(f"  GPU index build failed ({e}); continuing on CPU.")
            working_index = cpu_index
            gpu_res = None

    if use_ivf and not working_index.is_trained:
        # A representative sample is enough to place the coarse clusters -
        # training on the full 50L+ set would be slow and unnecessary.
        train_size = min(n, max(nlist * 40, IVF_TRAIN_SAMPLE_MIN))
        rng = np.random.default_rng(42)
        train_idx = rng.choice(n, size=train_size, replace=False)
        working_index.train(embeddings[train_idx])

    working_index.add(embeddings)

    if gpu_res is not None:
        working_index = faiss.index_gpu_to_cpu(working_index)  # always save a CPU index

    if use_ivf:
        working_index.nprobe = IVF_NPROBE

    kind = "ivf" if use_ivf else "flat"
    return working_index, kind, nlist


def build_index(input_dirs, name):
    index_file = OUT_DIR / f"{name}.index"
    mapping_file = OUT_DIR / f"{name}_mapping.parquet"
    info_file = OUT_DIR / f"{name}_info.json"

    if index_file.exists() and mapping_file.exists():
        print(f"[SKIP] {name}")
        return

    print(f"[BUILD] {name}")
    embeddings, ids = load_embeddings_and_ids(input_dirs)
    validate_embeddings(embeddings, name)

    # Defensive re-normalize - Step 2 already L2-normalizes before saving;
    # this is cheap insurance in case that step was skipped or changed.
    faiss.normalize_L2(embeddings)

    index, kind, nlist = build_faiss_index(embeddings, USE_GPU)

    faiss.write_index(index, str(index_file))
    pd.DataFrame({"entity_id": ids}).to_parquet(mapping_file, index=False)

    info = {
        "index_kind": kind,
        "nlist": nlist,
        "nprobe": IVF_NPROBE if kind == "ivf" else None,
        "dim": int(embeddings.shape[1]),
        "n_vectors": int(embeddings.shape[0]),
        "built_with_gpu": USE_GPU,
    }
    with open(info_file, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)

    nlist_note = f", nlist={nlist}" if nlist else ""
    print(f"  {name}: {len(ids):,} vectors indexed ({kind}{nlist_note})")

In [ ]:
# =====================================================
# RUN
# =====================================================

build_index(TRAIN_DIRS, "faiss_train")
build_index(TEST_DIRS, "faiss_test")

(OUT_DIR / "_SUCCESS").touch()
print("FAISS index build completed.")

## Step 3b (optional, not in your original script) — query the index

An index by itself doesn't produce candidates. This searches every Source 1 embedding against the
matching index (`faiss_train` for train-split S1 entities, `faiss_test` for test-split ones) and keeps
the top-`K` neighbors as your candidate set, per your Top-K candidate generation plan. Delete this
section if you already have a separate script for it.

Set `SOURCE1_TRAIN_EMBEDDINGS_DIR` / `SOURCE1_TEST_EMBEDDINGS_DIR` to wherever Step 2 saved the Source 1
embeddings (same notebook, run with `SOURCE_LABEL = "source1_train"` / `"source1_test"`).

In [ ]:
SOURCE1_TRAIN_EMBEDDINGS_DIR = Path("processed/source1_train_embeddings")
SOURCE1_TEST_EMBEDDINGS_DIR = Path("processed/source1_test_embeddings")

TOP_K = 20  # per-blocker top-K, unioned later with the TF-IDF and rule-based blockers


def search_index(name, query_dir, top_k):
    index_file = OUT_DIR / f"{name}.index"
    mapping_file = OUT_DIR / f"{name}_mapping.parquet"
    info_file = OUT_DIR / f"{name}_info.json"
    out_file = OUT_DIR / f"{name}_candidates.parquet"

    if out_file.exists():
        print(f"[SKIP] {name} search - candidates already exist")
        return

    index = faiss.read_index(str(index_file))
    mapping = pd.read_parquet(mapping_file)
    with open(info_file, "r", encoding="utf-8") as f:
        info = json.load(f)
    if info["index_kind"] == "ivf":
        index.nprobe = info["nprobe"]

    query_embeddings, query_ids = load_embeddings_and_ids([query_dir])
    faiss.normalize_L2(query_embeddings)

    scores, indices = index.search(query_embeddings, top_k)

    # Vectorized instead of a per-row Python loop - at millions of queries
    # x top_k, a plain nested loop with pandas .iloc lookups is the kind
    # of thing that quietly turns into the slowest part of the pipeline.
    mapping_ids = mapping["entity_id"].to_numpy()
    query_ids_arr = np.array(query_ids)

    valid_mask = indices != -1
    row_idx, col_idx = np.nonzero(valid_mask)

    candidates = pd.DataFrame({
        "source1_entity_id": query_ids_arr[row_idx],
        "candidate_entity_id": mapping_ids[indices[row_idx, col_idx]],
        "score": scores[row_idx, col_idx],
    })
    candidates.to_parquet(out_file, index=False)
    print(f"  {name}: {len(query_ids):,} queries -> {len(candidates):,} candidate pairs")


search_index("faiss_train", SOURCE1_TRAIN_EMBEDDINGS_DIR, TOP_K)
search_index("faiss_test", SOURCE1_TEST_EMBEDDINGS_DIR, TOP_K)